___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://sodal.cl/wp-content/uploads/2024/08/caracteristicas-aluminio-600x375.jpg" width="380px" height="200px" />


# <font color= #bbc28d> **Sistema Híbrido de Predicción y Gestión de Riesgo (FNN + GARCH)** </font>
#### <font color= #2E9AFE> `Examen 2 - MNLP`</font>
- <Strong> Diana Valdivia, Daniela de la Torre, Aissa Gonzáles & Rafael Takata. </Strong>
- <Strong> Fecha </Strong>: 12/04/2026.

___

<p style="text-align:right;"> Image retrieved from: https://sodal.cl/wp-content/uploads/2024/08/caracteristicas-aluminio-600x375.jpg</p>

El activo elegido para este estudio es el aluminio, un commodity industrial de gran relevancia por su amplia utilización en sectores como la construcción, el transporte, el empaque y la manufactura. Su precio está influenciado por factores como la oferta global, la demanda industrial, los costos energéticos y las condiciones macroeconómicas, lo que lo convierte en un activo interesante para analizar desde una perspectiva financiera y de riesgo. Además, al tratarse de un commodity con variaciones frecuentes en su precio, representa un buen caso de estudio para aplicar modelos predictivos y de volatilidad. La motivación principal de este trabajo es mostrar cómo un Hedge Fund podría usar herramientas FFNN y GARCH para apoyar sus decisiones de inversión.

In [31]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import acf, pacf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Renderizado HTML en Jupyter / VS Code
pio.renderers.default = "notebook"

In [44]:
datos = pd.read_csv(r"Aluminium Historical Data.csv")
datos = datos[['Date', 'Price']].copy()

datos['Date'] = pd.to_datetime(datos['Date'], format='mixed')
datos['Price'] = datos['Price'].str.replace(',', '').astype(float)

datos = datos.sort_values('Date')

fig = px.line(datos, x='Date', y='Price', title='Precio del Aluminio')
fig.show()

In [33]:
# Parsear fecha y ordenar cronológicamente
datos = datos.sort_values('Date').reset_index(drop=True)
datos.set_index('Date', inplace=True)

print("=" * 55)
print("  RESUMEN GENERAL DEL DATASET")
print("=" * 55)
print(f"  Periodo     : {datos.index.min().date()} → {datos.index.max().date()}")
print(f"  Observaciones: {len(datos):,}")
print(f"  Frecuencia aprox.: {pd.infer_freq(datos.index) or 'irregular (ver gaps)'}")

# Valores nulos
nulos = datos.isnull().sum()
print(f"  Valores nulos: {nulos.values[0]}")

  RESUMEN GENERAL DEL DATASET
  Periodo     : 2022-01-10 → 2026-04-02
  Observaciones: 1,070
  Frecuencia aprox.: irregular (ver gaps)
  Valores nulos: 0


In [34]:
fechas_completas = pd.date_range(datos.index.min(), datos.index.max())
faltantes = fechas_completas.difference(datos.index)

print(len(faltantes))
print(faltantes[:10])

474
DatetimeIndex(['2022-01-15', '2022-01-16', '2022-01-22', '2022-01-23',
               '2022-01-29', '2022-01-30', '2022-02-05', '2022-02-06',
               '2022-02-12', '2022-02-13'],
              dtype='datetime64[us]', freq=None)


In [36]:
# Rellenamos business days faltantes
datos = datos.asfreq('B').ffill()

fechas_completas = pd.date_range(datos.index.min(),datos.index.max(),freq='B')
faltantes = fechas_completas.difference(datos.index)

print(len(faltantes))

0


# **<font color= #bbc28d> GARCH </font>**
El modelo **GARCH (Generalized Autoregressive Conditional Heteroskedasticity)** es una técnica utilizada para modelar y predecir la **volatilidad** en series de tiempo financieras, como precios de commodities, acciones o divisas.

A diferencia de modelos tradicionales que asumen varianza constante, GARCH permite que la **varianza cambie en el tiempo**, lo cual es más realista para datos financieros donde existen periodos de alta y baja volatilidad.

### **Idea principal**
- Los retornos financieros suelen mostrar **clustering de volatilidad** 
- Periodos con alta volatilidad tienden a ser seguidos por alta volatilidad
- Periodos tranquilos tienden a permanecer tranquilos

GARCH modela este comportamiento haciendo que la varianza dependa de:
1. Errores pasados (shocks recientes)
2. Varianza pasada (persistencia)

### **¿Por qué usarlo en el precio del aluminio?**
El precio del aluminio, como otros commodities, presenta:
- Periodos de alta volatilidad
- Eventos macroeconómicos
- Choques de oferta y demanda

GARCH permite capturar estos cambios dinámicos en la volatilidad y generar predicciones más realistas.



## **<font color= #bbc28d> &ensp; • Obtenemos datos y gráficas de serie de tiempo </font>**
Obtenemos la serie de tiempo del sitio web ìnvesting.com` para graficar. Se calcula los rendimientos logarítmicos porcentuales, los cuales se representan como:$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) \times 100$$

Usualmente se usan retorno logaritmico porque en escalas pequeñas su compertamiento es muy similar al del retorno "normal" y al ser logaritmo, el comportamiento sigue una distribución más estable.

In [45]:
# Calculamos los retornos logarítmicos porcentuales usando 'Close'
returns = 100 * np.log(datos['Price'] / datos['Price'].shift(1)).dropna()

# Gráfica de la serie de retornos usando Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines', name='Retornos MAL3'))
fig.update_layout(title=f'Retornos Diarios del Aluminio',
                  yaxis_title='Retornos (%)')
fig.show()

## **<font color= #bbc28d> &ensp; • Selección y validación de modelo. </font>**

Para justificar si es estadísticamente viable utilizar un modelo GARCH, se calculan las graficas ACF y PACF; pero utilizando los retornos al cuadrado.

**Retornos al Cuadrado**

Las gráficas ACF y PACF miden la autocorrelación (qué tanto se parece el dato de hoy vs los datos del pasado)
- ACF/PACF de los Retornos al Cuadrado ($r_t^2$)
  - ¿Qué miden? ->
    Dependencia no lineal en el segundo momento estadístico (la varianza/volatilidad). Al elevar al cuadrado, eliminamos el signo (positivo o negativo) y nos quedamos solo con la magnitud del movimiento.
  - ¿Qué pregunta responden? ->
    Saber que ayer hubo un movimiento violento (sin importar si fue hacia arriba o hacia abajo) me ayuda a predecir si hoy el mercado seguirá turbulento.
  - Lo que verás en la gráfica ->
    Verás múltiples barras que sobresalen de la zona de significancia. Esto demuestra matemáticamente el Volatility Clustering (agrupamiento de volatilidad).

Si la PACF o ACF de retornos al cuadrado no muestra barras significativas, no se recomienda utilizar un modelo GARCH (Puede ser un modelo EGARCH o de otra familia)

In [46]:
# Retornos al cuadrado
sq_returns = returns**2

# Calcular ACF y PACF
lag_acf = acf(sq_returns, nlags=20)
lag_pacf = pacf(sq_returns, nlags=20, method='ols')

# Graficar ACF y PACF con Plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('ACF de Retornos al Cuadrado', 'PACF de Retornos al Cuadrado'))

# Añadir barras de ACF
fig.add_trace(go.Bar(x=np.arange(len(lag_acf)), y=lag_acf, name='ACF'), row=1, col=1)
# Añadir barras de PACF
fig.add_trace(go.Bar(x=np.arange(len(lag_pacf)), y=lag_pacf, name='PACF'), row=1, col=2)

# Líneas de significancia (aprox 1.96 / sqrt(N))
sig_level = 1.96 / np.sqrt(len(returns))
for i in [1, 2]:
    fig.add_hline(y=sig_level, line_dash="dash", line_color="red", row=1, col=i)
    fig.add_hline(y=-sig_level, line_dash="dash", line_color="red", row=1, col=i)

fig.update_layout(title='Análisis de Dependencia de Varianza', showlegend=False)
fig.show()

## **<font color= #bbc28d> &ensp; • Modelado </font>**

### **Forma general del modelo GARCH(1,1)**

La varianza condicional se define como:

$
\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2
$

Donde:
- $ \sigma_t^2 $ : varianza condicional en el tiempo t
- $ \omega $ : constante
- $ \alpha $ : efecto de shocks recientes
- $ \beta $ : persistencia de la volatilidad
- $ \varepsilon_{t-1}^2 $ : error cuadrado del periodo anterior

In [ ]:
# Aqui va codigo

# **<font color= #66b0b0> FFNN </font>**
Una **FFNN (Feedforward Neural Network)** es un tipo de red neuronal artificial utilizada para modelar relaciones no lineales entre variables. Se denomina *feedforward* porque la información fluye en una sola dirección: desde la capa de entrada, pasando por capas ocultas, hasta la capa de salida, sin ciclos ni retroalimentación.

Estas redes son especialmente útiles cuando la relación entre variables es compleja y difícil de modelar con técnicas estadísticas tradicionales.

### **¿Para qué se usa en series financieras?**

Las FFNN pueden:

* Capturar relaciones no lineales 
* Detectar patrones complejos
* Aprender dinámicas temporales usando *lags*

### **Aplicación al precio del aluminio**

Para el precio del aluminio, una FFNN puede:

* Aprender patrones ocultos en la serie
* Modelar dinámicas no lineales del mercado



In [ ]:
# Aqui va codigo

## <font color= #66b0b0> &ensp; • **Modelado** </font>
Una red neuronal feedforward se compone de:

* **Capa de entrada**: recibe los datos (por ejemplo, retornos pasados)
* **Capas ocultas**: transforman la información mediante funciones no lineales
* **Capa de salida**: genera la predicción final

Cada neurona aplica la siguiente transformación:

$
y = f\left(\sum_{i=1}^{n} w_i x_i + b \right)
$ 

Donde:

* $x_i$ : variables de entrada
* $w_i$ : pesos de la red
* $b$ : sesgo (bias)
* $f$ : función de activación (ReLU, tanh, sigmoid, etc.)

In [ ]:
# Aqui va codigo